# Init

In [0]:
from pyspark.sql import functions as F

# Reading from Bronze Table

In [0]:
df = spark.table("aerovista_bootcamp.bronze.travel_trips")

display(df)

# Transformation

## adding date and time columns

In [0]:
df_updated = df \
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime")) \
    .withColumn("pickup_time", F.date_format("tpep_pickup_datetime", "HH:mm:ss")) \
    .withColumn("dropoff_date", F.to_date("tpep_dropoff_datetime")) \
    .withColumn("dropoff_time", F.date_format("tpep_dropoff_datetime", "HH:mm:ss"))



## drop original datetime column

In [0]:
df_updated = df_updated.drop("tpep_pickup_datetime", "tpep_dropoff_datetime")

## moving the date and time columns to the beginning of the table

In [0]:
new_columns = ["pickup_date", "pickup_time", "dropoff_date", "dropoff_time"]
remaining_columns = [col for col in df_updated.columns if col not in new_columns]

df_updated = df_updated.select(new_columns + remaining_columns)

## check for null values

In [0]:
from pyspark.sql.types import DoubleType, FloatType

null_counts = df_updated.select([
    F.count(F.when(
        F.col(c).isNull() | (F.isnan(c) if isinstance(df_updated.schema[c].dataType, (DoubleType, FloatType)) else F.lit(False)),
        c
    )).alias(c) 
    for c in df_updated.columns
])

display(null_counts)

In [0]:
display(df_updated)

# Write into Silver Table

In [0]:
df_updated.write.mode("overwrite").saveAsTable("aerovista_bootcamp.silver.travel_trips")